# Aula 10 — Laboratório: Curvas de validação

**Consultoria Especializada em Ciência de Dados 1**

Na Aula 09 a busca em grade devolveu a melhor combinação de hiperparâmetros. Hoje a gente abre a
caixa: fixa o modelo inteiro, varia **um** hiperparâmetro e olha o que acontece com o desempenho
no treino e na validação.

O cenário é clínico: prever quais pacientes são de **alto risco metabólico** (cerca de 15% da base).
Vamos usar duas curvas — a profundidade de uma árvore e o `C` de uma regressão logística — e, no
fim, comparar a curva com a busca em grade.

## Parte 0 — Ambiente, dados e a função de desenho

Carregamos os pacientes, montamos o alvo binário e criamos os **dois pipelines** do laboratório.
O `StandardScaler` vem antes do modelo: dentro da validação cruzada, a padronização é ajustada
**só na parte de treino** de cada dobra, sem vazamento.

A função `plota_curva` desenha média ± desvio das duas curvas e marca o máximo da validação.
Rode as duas células e siga.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import validation_curve, StratifiedKFold, GridSearchCV

pac = pd.read_csv("datasets/1_pacientes.csv")
pac["alto_risco"] = (pac["risco_metabolico"] == "alto").astype(int)
feat = [c for c in pac.columns if c not in ["id", "risco_metabolico", "alto_risco", "sexo"]]
X, y = pac[feat], pac["alto_risco"]

arvore    = Pipeline([("scaler", StandardScaler()),
                      ("clf", DecisionTreeClassifier(random_state=42))])
logistica = Pipeline([("scaler", StandardScaler()),
                      ("clf", LogisticRegression(max_iter=1000))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # as MESMAS dobras sempre

print("pacientes:", len(y), "| variáveis:", len(feat))
print("alto risco:", int(y.sum()), f"({y.mean():.1%})")
print("linha de base (dizer 'ninguém é de alto risco'):", round(1 - y.mean(), 4))

In [ ]:
def plota_curva(valores, tr, va, nome, log=False):
    """tr e va são as MATRIZES devolvidas por validation_curve (valor x dobra)."""
    tr_m, tr_s = tr.mean(axis=1), tr.std(axis=1)
    va_m, va_s = va.mean(axis=1), va.std(axis=1)
    plt.figure(figsize=(8, 4.5))
    plt.plot(valores, tr_m, "o-", color="#0F766E", label="treino")
    plt.fill_between(valores, tr_m - tr_s, tr_m + tr_s, alpha=.15, color="#0F766E")
    plt.plot(valores, va_m, "s-", color="#C2410C", label="validação")
    plt.fill_between(valores, va_m - va_s, va_m + va_s, alpha=.15, color="#C2410C")
    melhor = valores[int(np.argmax(va_m))]
    plt.axvline(melhor, color="#9F1239", ls="--", label=f"máximo da validação = {melhor}")
    if log:
        plt.xscale("log")
    plt.xlabel(nome); plt.ylabel("desempenho na CV (5 dobras)")
    plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()
    return melhor

## Parte 1 — A primeira curva de validação: a profundidade da árvore

O `max_depth` controla a **complexidade** da árvore: quanto maior, mais ela pode decorar o treino.
Dentro de um Pipeline esse parâmetro se chama **`clf__max_depth`** — nome do passo, dois
sublinhados, nome do parâmetro.

A `validation_curve` faz, para **cada** valor da lista, uma validação cruzada completa, e devolve
duas **matrizes**: uma linha por valor testado, uma coluna por dobra.

### Exemplo — três profundidades, para ver o formato da saída

Antes de varrer tudo, olhe o que a função devolve com apenas três valores.

In [ ]:
tr3, va3 = validation_curve(
    arvore, X, y,
    param_name="clf__max_depth",
    param_range=[1, 4, 15],
    cv=cv, scoring="accuracy")

print("formato das matrizes:", tr3.shape, "-> 3 valores x 5 dobras")
for p, t, v in zip([1, 4, 15], tr3.mean(axis=1), va3.mean(axis=1)):
    print(f"prof {p:2d}   treino {t:.4f}   validação {v:.4f}   gap {t - v:.4f}")

### Exercício 1 — a curva completa

Refaça a chamada para as profundidades de **1 a 15** e desenhe o resultado com `plota_curva`.

In [ ]:
profundidades = list(range(1, 16))

# TODO: gere a curva de validação do pipeline 'arvore' variando 'clf__max_depth'
tr, va = ____

m_tr, m_va, s_va = tr.mean(axis=1), va.mean(axis=1), va.std(axis=1)
plota_curva(profundidades, tr, va, "profundidade da árvore (clf__max_depth)")

## Parte 2 — As três regiões

Toda curva bem-comportada tem três regiões:

1. **Subajuste** (esquerda) — treino **e** validação baixos, gap pequeno. O modelo é simples demais.
2. **Ótimo** (meio) — a validação atinge o máximo.
3. **Sobreajuste** (direita) — o treino sobe rumo a 1,00, a validação **não melhora mais**, e o
   **gap** (treino − validação) cresce.

Atenção à definição: **gap = treino − validação, no mesmo valor do hiperparâmetro**. Não é a
diferença entre dois valores do hiperparâmetro, nem entre validação e teste.

### Exemplo — a tabela das três profundidades da Parte 1

In [ ]:
exemplo = pd.DataFrame({"profundidade": [1, 4, 15],
                        "treino":     tr3.mean(axis=1).round(4),
                        "validação":  va3.mean(axis=1).round(4)})
exemplo["gap"] = (exemplo["treino"] - exemplo["validação"]).round(4)
exemplo["leitura"] = ["subajuste", "ótimo", "sobreajuste"]
exemplo

### Exercício 2 — a tabela completa

Monte um `DataFrame` com as 15 profundidades, as colunas `treino`, `validação`, `desvio` e `gap`,
e ordene **pela validação**, da maior para a menor.

In [ ]:
tabela = pd.DataFrame({"profundidade": profundidades,
                       "treino":    m_tr.round(4),
                       "validação": m_va.round(4),
                       "desvio":    s_va.round(4)})

# TODO: crie a coluna 'gap' (treino - validação) e ordene pela validação, decrescente
tabela["gap"] = ____
tabela = ____
tabela.head(6)

## Parte 3 — Escolher o valor e medir o gap

A régua é sempre a mesma: **o valor que maximiza a validação**, nunca o que maximiza o treino —
esse levaria sempre ao modelo mais complexo.

E sempre que você reportar um valor escolhido, reporte **o gap naquele ponto**. É a medida mais
direta de sobreajuste, e quase ninguém põe no relatório.

### Exemplo — achar o máximo num vetor pequeno

`argmax` devolve a **posição** do maior valor, não o valor em si. É dela que sai o índice usado
para buscar o treino, o desvio e o hiperparâmetro correspondentes.

In [ ]:
demo = np.array([0.80, 0.86, 0.84])
i_demo = int(np.argmax(demo))
print("posição do máximo:", i_demo, "| valor:", demo[i_demo])

### Exercício 3 — a escolha e o gap

Encontre a profundidade que **maximiza a validação** e imprima, nesse ponto, a validação, o treino,
o desvio e o gap. Imprima também o gap na profundidade 15, para comparar.

In [ ]:
# TODO: posição do máximo da validação
i = ____

melhor_prof = profundidades[i]
print("melhor profundidade :", melhor_prof)
print("validação           :", round(m_va[i], 4))
print("treino              :", round(m_tr[i], 4))
print("desvio da validação :", round(s_va[i], 4))
print("gap no ponto escolhido:", round(____, 4))
print("gap na profundidade 15:", round(m_tr[-1] - m_va[-1], 4))

## Parte 4 — O topo é um pico ou um platô?

O desvio-padrão entre as dobras diz o quanto aquele número balança. Se a diferença entre dois
valores do hiperparâmetro é **menor que o desvio**, eles empatam dentro do erro — e aí a escolha
passa a ser por outro critério: o modelo **mais simples**.

### Exemplo — comparar dois valores usando o desvio

In [ ]:
a, b = 4, 2        # as duas profundidades a comparar
ia, ib = profundidades.index(a), profundidades.index(b)
dif = m_va[ia] - m_va[ib]
print(f"prof {a}: {m_va[ia]:.4f} ± {s_va[ia]:.4f}")
print(f"prof {b}: {m_va[ib]:.4f} ± {s_va[ib]:.4f}")
print(f"diferença = {dif:.4f}   |   desvio do melhor = {s_va[ia]:.4f}")
print("empatam dentro do erro?", dif < s_va[ia])

### Exercício 4 — o desempate

Calcule o **piso** da faixa de empate (máximo da validação menos um desvio desse ponto), liste
todas as profundidades que ficam acima dele e escolha a **mais simples** da faixa. Mostre a
validação e o gap da escolhida.

In [ ]:
# TODO: piso = máximo da validação menos o desvio DESSE ponto
piso = ____
dentro = [p for p, v in zip(profundidades, m_va) if v >= piso]

escolhida = ____          # TODO: a mais simples da faixa
j = profundidades.index(escolhida)
print("piso da faixa    :", round(piso, 4))
print("dentro da faixa  :", dentro)
print("escolhida        :", escolhida)
print("validação / gap  :", round(m_va[j], 4), "/", round(m_tr[j] - m_va[j], 4))

## Parte 5 — Segundo eixo: o `C` da regressão logística

O `C` controla a **regularização**. Regularização é um freio: além de errar pouco no treino, o
modelo é obrigado a manter os coeficientes pequenos — coeficiente grande só se valer a pena.

O `C` multiplica o **termo do erro** na conta, e não a penalidade. Por isso ele é invertido:
`C` **pequeno** = freio forte = modelo **simples**; `C` **grande** = freio fraco = modelo flexível.

Como o `C` varia por ordens de grandeza, os valores são espaçados em escala **logarítmica** — e o
eixo do gráfico também.

### Exemplo — por que o eixo precisa ser log

Veja onde caem os valores em escala linear: tudo abaixo de 1 vira um borrão colado no zero.

In [ ]:
Cs = np.logspace(-4, 3, 8)
print("valores de C:", Cs)

fig, ax = plt.subplots(1, 2, figsize=(9, 2.2))
for eixo, log in zip(ax, [False, True]):
    eixo.plot(Cs, np.zeros_like(Cs), "o", color="#0F766E")
    if log:
        eixo.set_xscale("log")
    eixo.set_title("escala log" if log else "escala linear")
    eixo.set_yticks([])
plt.tight_layout(); plt.show()

### Exercício 5 — a curva do `C`

Gere a curva de validação do pipeline `logistica` variando `clf__C` nesses 8 valores e desenhe com
`plota_curva`, lembrando do `log=True`.

In [ ]:
# TODO: gere a curva de validação do 'logistica' variando 'clf__C'
tr_c, va_c = ____

melhor_C = plota_curva(Cs, tr_c, va_c, "C da regressão logística", log=True)
print("melhor C:", melhor_C, "| validação:", round(va_c.mean(axis=1).max(), 4))

## Parte 6 — O ponto cego da acurácia

A base tem cerca de 15% de positivos. Numa base assim, a acurácia tem um ponto cego: quem responde
"ninguém é de alto risco" já acerta 0,8494 sem olhar para nenhum paciente.

### Exemplo — o mesmo modelo, duas réguas

Olhe o primeiro ponto da curva do `C` (o modelo mais regularizado de todos) medido das duas
maneiras.

In [ ]:
tr_f, va_f = validation_curve(
    logistica, X, y,
    param_name="clf__C", param_range=Cs,
    cv=cv, scoring="f1")

print(f"em C = 0,0001  ->  acurácia {va_c.mean(axis=1)[0]:.4f}   F1 {va_f.mean(axis=1)[0]:.4f}")

### Exercício 6 — a curva em F1

Desenhe a curva do `C` medida em **F1** e compare: o `C` escolhido muda? E o que muda na leitura
da região de subajuste?

In [ ]:
melhor_C_f1 = plota_curva(Cs, tr_f, va_f, "C da regressão logística (F1)", log=True)

# TODO: imprima, lado a lado, a validação em acurácia e em F1 para cada C
for c, a, f in zip(Cs, ____, ____):
    print(f"C = {c:<8g}  acurácia {a:.4f}   F1 {f:.4f}")

## Parte 7 — A curva e a busca em grade

Uma pergunta justa: se o `GridSearchCV` já varre vários valores, para que serve a curva?

Com **um** hiperparâmetro só, a busca com `return_train_score=True` devolve os mesmos números — aí a
curva é conveniência. A diferença aparece com **vários** eixos: o `cv_results_` tem uma linha por
*combinação*, e o efeito de um eixo fica misturado com os outros.

### Exemplo — a busca no mesmo eixo da Parte 1

Repare no `return_train_score=True`: sem ele, a busca **não guarda** o score de treino — e sem
treino não existe gap.

In [ ]:
busca = GridSearchCV(arvore, {"clf__max_depth": profundidades},
                     cv=cv, scoring="accuracy",
                     return_train_score=True).fit(X, y)

print("best_params_:", busca.best_params_)
print("best_score_ :", round(busca.best_score_, 4))

### Exercício 7 — os mesmos números?

Extraia do `cv_results_` as médias de treino e de validação e verifique se batem com as da curva da
Parte 1.

In [ ]:
res = pd.DataFrame(busca.cv_results_)

# TODO: compare as médias do cv_results_ com m_tr e m_va
print("validação bate?", np.allclose(res[____].values, m_va))
print("treino bate?   ", np.allclose(res[____].values, m_tr))

---

**Parabéns!** Você construiu duas curvas de validação, leu as três regiões, mediu o gap, usou o
desvio para desempatar e viu o ponto cego da acurácia.

Para levar ao projeto integrado: rode a curva nos dois ou três hiperparâmetros que importam **antes**
da busca — é ela que define a faixa que vale procurar — e, no relatório, informe o valor escolhido
**com o gap naquele ponto**.

Na **próxima aula**: curvas de **aprendizado**, em que o eixo x passa a ser o tamanho do conjunto de
treino. A pergunta vira: mais dados de treino sempre ajudam?